# Notely AI Agent Tool Layer

This notebook explains the local Python tool layer that will eventually sit underneath the AI agent.

At this stage, there is no LLM yet. That is intentional.

The purpose of this layer is to create deterministic, testable functions for:

- listing tables
- describing table schemas
- safely running read-only SQL
- retrieving trusted metrics
- retrieving weekly report data
- retrieving product context

Later, OpenAI function calling can expose these same functions to the model.

## 1. Why We Need a Tool Layer

The metric layer answers: **how do we calculate metrics?**

The tool layer answers: **what actions can the future agent safely perform?**

Instead of letting an LLM directly touch the database, we give it controlled tools such as:

```text
get_metric("activation_rate", start_date, end_date, group_by="platform")
get_weekly_report("2026-06-01")
get_product_context("2026-05-01", "2026-06-05")
run_sql("SELECT ...")
```

This makes the agent safer, easier to test, and easier to explain.

## 2. Import Tools

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

import sys
sys.path.append(str(project_root))

from agent.tools import (
    ToolError,
    available_metrics,
    describe_table,
    get_metric,
    get_product_context,
    get_weekly_report,
    list_tables,
    run_sql,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

## 3. Tool: `list_tables()`

This tool gives the agent a high-level map of the database.

In [ ]:
pd.DataFrame(list_tables())

## 4. Tool: `describe_table(table_name)`

This tool gives the agent schema details for one table at a time.

This is important because we do not want to stuff the entire database schema into every prompt forever. Later, the agent can inspect only the tables it needs.

In [ ]:
events_schema = describe_table("events")
events_schema

In [ ]:
pd.DataFrame(events_schema["columns"])

## 5. Tool: `run_sql(sql)`

This tool runs ad hoc read-only SQL.

It is intentionally guarded:

- only `SELECT` or `WITH` queries are allowed
- write operations are blocked
- multi-statement SQL is blocked
- a result limit is applied automatically

In [ ]:
result = run_sql("""
SELECT
  event_name,
  COUNT(*) AS events
FROM events
GROUP BY 1
ORDER BY events DESC
""", limit=10)

pd.DataFrame(result["rows"])

### Safety Check

A future PM-facing agent should not be able to drop or mutate tables. This test shows that unsafe SQL gets blocked.

In [ ]:
try:
    run_sql("DROP TABLE users")
except ToolError as exc:
    print("Blocked unsafe SQL:", exc)

## 6. Tool: `available_metrics()`

This tells the future agent which trusted metrics are currently supported.

In [ ]:
available_metrics()

## 7. Tool: `get_metric(...)`

This is where the metric layer starts becoming callable.

Instead of asking the model to invent SQL for activation rate, we give it a function with a controlled interface.

In [ ]:
activation = get_metric(
    metric_name="activation_rate",
    start_date="2026-05-01",
    end_date="2026-05-31",
    group_by="platform",
)

pd.DataFrame(activation["rows"])

In [ ]:
paid_conversion = get_metric(
    metric_name="paid_conversion_rate",
    start_date="2026-06-01",
    end_date="2026-06-26",
    group_by="acquisition_channel",
)

pd.DataFrame(paid_conversion["rows"])

## 8. Tool: `get_product_context(...)`

This retrieves launches, incidents, pricing changes, experiments, and campaigns from `product_calendar`.

Later this can evolve into RAG over richer docs like release notes, experiment readouts, and incident logs.

In [ ]:
context = get_product_context("2026-05-01", "2026-06-05")
pd.DataFrame(context["rows"])

## 9. Tool: `get_weekly_report(...)`

This wraps the weekly PM report SQL prototype.

The future Report Agent can call this function, then summarize the output in natural language.

In [ ]:
weekly_report = get_weekly_report("2026-06-01")
pd.DataFrame(weekly_report["rows"])

## 10. How This Maps to Future Function Calling

Future flow:

```text
PM asks: "Why did activation drop in mid-May?"
        ->
LLM decides it needs tools
        ->
Calls get_metric("activation_rate", ..., group_by="platform")
Calls get_metric("upload_completion_rate", ..., group_by="platform")
Calls get_product_context("2026-05-01", "2026-05-22")
        ->
Python tools query SQLite safely
        ->
LLM receives structured results
        ->
LLM writes an evidence-backed answer
```

So the tool layer is the bridge between the metric layer and the future AI agent.